In [ ]:
# ============================================================
# Cell 1: Environment Setup & GPU Detection
# ============================================================
import os, sys, gc, json, time
os.chdir(os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd())
if os.path.basename(os.getcwd()) != "nst":
    for p in ['.', '..']:
        if os.path.exists(os.path.join(p, 'training', 'train_fever_veri.py')):
            os.chdir(p); break
sys.path.insert(0, os.getcwd())

import torch
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE.upper()

print(f"{'='*60}")
print(f"  NST-VERI Rapid Iteration — {DEVICE_NAME}")
print(f"  PyTorch {torch.__version__}")
print(f"  CUDA: {torch.version.cuda}" if DEVICE == "cuda" else "")
if DEVICE == "cuda":
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"  GPU Memory: {mem:.1f} GB")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 1: Environment Setup & GPU Detection
# ============================================================
import os, sys, time, gc, pathlib

# ── Find project root ──
# Strategy: look for data/ and models/ directories to identify root.
# Works in Colab (/content/nst), local dev, or any layout.
_candidates = [
    os.getcwd(),
    os.path.dirname(os.path.abspath("__file__")),  # notebook dir (local)
    "/content/nst",       # Colab default after git clone
    "/content/Neurosymbolic-Transformers",
    "/content",
]
PROJ_ROOT = None
for c in _candidates:
    if os.path.isdir(os.path.join(c, "data")) and os.path.isdir(os.path.join(c, "models")):
        PROJ_ROOT = c
        break

if PROJ_ROOT is None:
    # If repo isn't cloned yet (Colab), clone it
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git",
                    "/content/nst"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/nst"], check=True)
    PROJ_ROOT = "/content/nst"

if PROJ_ROOT not in sys.path:
    sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)
print(f"Project root: {PROJ_ROOT}")

# Verify core imports
import torch
import transformers
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")

# ═══════════════════════════════════════════════════════════
#  GPU Auto-Detection
# ═══════════════════════════════════════════════════════════
GPU_OVERRIDES = {}

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    cc = (props.major, props.minor)
    supports_bf16 = cc >= (8, 0)

    if vram_gb >= 35:
        BS, GA = 32, 2
    elif vram_gb >= 20:
        BS, GA = 24, 2
    else:
        BS, GA = 16, 2

    GPU_OVERRIDES = {
        "train": {
            "batch_size": BS,
            "grad_accum_steps": GA,
            "bf16": supports_bf16,
            "fp16": not supports_bf16,
            "tf32": supports_bf16,
            "fused_optimizer": True,
            "num_workers": 4,
        }
    }
    if supports_bf16:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    DEVICE = "cuda"
    DEVICE_NAME = f"{gpu_name} ({vram_gb:.0f}GB)"
    prec = "BF16" if supports_bf16 else "FP16"

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    # Apple Silicon — shared memory, smaller batches
    import subprocess
    r = subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True)
    ram_gb = int(r.stdout.strip()) / 1e9
    r2 = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True)
    chip = r2.stdout.strip()

    if ram_gb >= 32:
        BS, GA = 16, 2
    elif ram_gb >= 16:
        BS, GA = 8, 4
    else:
        BS, GA = 4, 8

    GPU_OVERRIDES = {
        "train": {
            "batch_size": BS,
            "grad_accum_steps": GA,
            "bf16": False,
            "fp16": True,
            "num_workers": 0,
        }
    }

    DEVICE = "mps"
    DEVICE_NAME = f"{chip} ({ram_gb:.0f}GB shared)"
    prec = "FP16"
else:
    DEVICE = "cpu"
    DEVICE_NAME = "CPU (no GPU)"
    BS, GA = 4, 8
    prec = "FP32"
    GPU_OVERRIDES = {
        "train": {
            "batch_size": BS,
            "grad_accum_steps": GA,
            "bf16": False,
            "fp16": False,
            "num_workers": 0,
        }
    }

eff_bs = BS * GA
print(f"\n{'='*60}")
print(f"  Device    : {DEVICE_NAME}")
print(f"  Backend   : {DEVICE}")
print(f"  Batch     : {BS} x {GA} = {eff_bs} effective")
print(f"  Precision : {prec}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 2: Verify datasets version is compatible
# ============================================================
# datasets 2.21.0 should already be installed (was installed above
# with kernel restart). Verify it.
import datasets
print(f"datasets version: {datasets.__version__}")
_ds_major = int(datasets.__version__.split(".")[0])
assert _ds_major < 3, (
    f"datasets {datasets.__version__} doesn't support FEVER loading scripts. "
    "Run: pip install datasets==2.21.0 then restart the kernel."
)
print("OK — compatible with FEVER loading script")

In [ ]:
# ============================================================
# Cell 3: Build Wiki Cache & Load Data
# ============================================================
# The FEVER dataset needs a wiki page cache to resolve evidence
# text from page title + sentence index. Without it, we only
# get page titles as "evidence", which cripples NLI training.
import logging, os, time
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

# Clear stale project modules
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

# ── Step 1: Build wiki cache if missing ──
from data.fever_wiki_cache import build_wiki_cache, cache_stats

cache_path = os.path.join(PROJ_ROOT, "data", "fever_wiki.db")
stats = cache_stats(cache_path)
if stats.get("exists"):
    print(f"Wiki cache exists: {stats['n_pages']} pages, {stats['size_mb']:.1f} MB")
else:
    print("Building wiki page cache (one-time, ~5-10 min on Colab)...")
    print("  This downloads ~1.7GB of Wikipedia pages and indexes ~25K needed pages.\n")
    t0 = time.time()
    build_stats = build_wiki_cache(cache_path=cache_path)
    elapsed = time.time() - t0
    print(f"\n  Done in {elapsed:.0f}s: {build_stats['n_found']}/{build_stats['n_needed']} pages "
          f"({build_stats['n_missing']} missing)")
    print(f"  Cache: {cache_path} ({build_stats['cache_size_mb']:.1f} MB)")

# ── Step 2: Load data with evidence text ──
from data.fever_dataset import load_fever_splits, print_fever_stats

splits_check = load_fever_splits(max_train=500, max_dev=200, dev_test_ratio=0.1, seed=42)
print_fever_stats(splits_check)

train_items = splits_check["train"]
if len(train_items) == 0:
    print("\n  ERROR: No training data loaded!")
else:
    n_with_evidence = sum(1 for it in train_items if len(it.get("gold_evidence_text", "")) > 30)
    pct = 100 * n_with_evidence / len(train_items)
    print(f"\n  Evidence quality: {n_with_evidence}/{len(train_items)} ({pct:.0f}%) have >30 char evidence")

    for i in [0, 1, 2]:
        it = train_items[i]
        ev = it.get("gold_evidence_text", it.get("evidence", ""))[:150]
        print(f"\n  [{i}] {it['claim'][:80]}")
        print(f"      Label: {it['label']}")
        print(f"      Evidence: {ev}")

    if pct > 60:
        print(f"\n  Data check PASSED — good evidence coverage")
    elif pct > 20:
        print(f"\n  Data check WARNING — partial evidence ({pct:.0f}%)")
    else:
        print(f"\n  Data check FAILED — only {pct:.0f}% have evidence. Wiki cache needed.")

In [ ]:
# ============================================================
# Cell 4: Smoke Test — 200 examples (validates pipeline end-to-end)
# ============================================================
import time, gc, json

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 60)
print("  SMOKE TEST: 200 train / 100 dev / 1 epoch")
print(f"  Device: {DEVICE_NAME}")
print("=" * 60 + "\n")

# Use same precision as real training to validate that path
_use_bf16 = GPU_OVERRIDES.get("train", {}).get("bf16", False)
_use_fp16 = GPU_OVERRIDES.get("train", {}).get("fp16", False)

SMOKE_OVERRIDES = {
    "data": {"max_train": 200, "max_dev": 100},
    "train": {
        "epochs": 1,
        "batch_size": min(BS, 16),
        "grad_accum_steps": 1,
        "eval_every_steps": 50,
        "patience": 99,
        "bf16": _use_bf16,
        "fp16": _use_fp16 and not _use_bf16,
        "num_workers": 0 if DEVICE == "mps" else 2,
    },
    "model": {"max_length": 128, "gradient_checkpointing": False},
    "io": {"out_dir": "outputs_smoke"},
}

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_smoke = train_fever_nst("configs/fever_gold_neural.yaml",
                                 config_overrides=SMOKE_OVERRIDES)
elapsed = time.time() - t0

dev = results_smoke.get("dev", {})
print(f"\n{'='*60}")
print(f"  SMOKE TEST RESULTS ({elapsed:.0f}s)")
print(f"{'='*60}")
print(f"  dev_accuracy  : {dev.get('accuracy', 'N/A')}")
print(f"  nan_abort     : {results_smoke.get('nan_abort', False)}")
print(f"  trainable     : {results_smoke.get('trainable_params_M', '?')}M / {results_smoke.get('total_params_M', '?')}M")

if results_smoke.get("nan_abort", False):
    print("\n  SMOKE TEST FAILED — NaN detected!")
else:
    print("\n  Smoke test PASSED — pipeline is working")

with open("results_smoke.json", "w") as f:
    json.dump(results_smoke, f, indent=2, default=str)

gc.collect()

In [ ]:
# ============================================================
# Cell 4: Neural Baseline — DeBERTa-v3-base (gold evidence)
# ============================================================
# Pure neural NLI baseline. No symbolic constraints.
# DeBERTa-v3-base on full (or subset) FEVER.
# This establishes the baseline accuracy to beat.
import time, json, gc

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Choose dataset size based on device ──
# MPS/16GB: use 10K subset for reasonable training time (~20-30 min)
# CUDA: use full dataset
if DEVICE == "mps":
    MAX_TRAIN = 10000
    MAX_DEV = 2000
    EPOCHS = 3
    EVAL_EVERY = 200
    est_time = "~20-30 min"
elif DEVICE == "cuda":
    MAX_TRAIN = None  # full ~145K
    MAX_DEV = None
    EPOCHS = 3
    EVAL_EVERY = 500
    est_time = "~25-40 min"
else:
    MAX_TRAIN = 2000
    MAX_DEV = 500
    EPOCHS = 2
    EVAL_EVERY = 100
    est_time = "~30-60 min"

BASELINE_OVERRIDES = {
    "data": {"max_train": MAX_TRAIN, "max_dev": MAX_DEV},
    "train": {
        "epochs": EPOCHS,
        "eval_every_steps": EVAL_EVERY,
        **GPU_OVERRIDES.get("train", {}),
    },
    "io": {"out_dir": "outputs_fever_neural_baseline"},
}
# Ensure correct model for memory-constrained devices
if DEVICE in ("mps", "cpu"):
    BASELINE_OVERRIDES["model"] = {
        "name": "microsoft/deberta-v3-base",
        "use_lora": False,
        "gradient_checkpointing": True,
        "max_length": 384,
    }

n_train_str = f"{MAX_TRAIN//1000}K" if MAX_TRAIN else "full (~145K)"
print("=" * 65)
print("  NEURAL BASELINE: DeBERTa-v3-base")
print("  Gold Evidence, Setting A")
print("=" * 65)
print(f"  Device    : {DEVICE_NAME}")
print(f"  Train set : {n_train_str}")
print(f"  Epochs    : {EPOCHS}")
print(f"  Batch     : {BASELINE_OVERRIDES['train'].get('batch_size', BS)} x "
      f"{BASELINE_OVERRIDES['train'].get('grad_accum_steps', GA)}")
print(f"  Est time  : {est_time}")
print("=" * 65 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_baseline = train_fever_nst("configs/fever_gold_neural.yaml",
                                    config_overrides=BASELINE_OVERRIDES)
elapsed_baseline = time.time() - t0

dev = results_baseline.get("dev", {})
dt = results_baseline.get("dev_test", {})
print(f"\n{'='*65}")
print(f"  NEURAL BASELINE RESULTS ({elapsed_baseline/60:.1f} min)")
print(f"{'='*65}")
print(f"  Train size : {n_train_str}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
if dt:
    print(f"  DevTest acc: {dt.get('accuracy', 'N/A')}  (held-out)")
print(f"  Best dev   : {results_baseline.get('best_dev_acc', 'N/A')}")
print(f"  Temperature: {results_baseline.get('temperature', 'N/A')}")
print(f"\n  Per-label (dev):")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_baseline.json", "w") as f:
    json.dump(results_baseline, f, indent=2, default=str)
print(f"\n  Saved to results_baseline.json")

gc.collect()

In [ ]:
# ============================================================
# Cell 6: NST-VERI — Neurosymbolic DeBERTa + CEGIS verification
# ============================================================
# Full neurosymbolic pipeline: DeBERTa backbone + constraint head
# + CEGIS counterexample loop. This is the novel contribution.
# Uses DeBERTa-v3-large + LoRA (vs base for baseline).
import time, json, gc

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# NST-VERI uses its own epoch count (5 = 3 phases) and eval settings.
# We only override GPU hardware settings and data limits.
VERI_OVERRIDES = {
    "data": {"max_train": MAX_TRAIN, "max_dev": MAX_DEV},
    "train": {
        # Use config's epochs (5) — needed for 3-phase training
        **{k: v for k, v in GPU_OVERRIDES.get("train", {}).items()
           if k not in ("epochs",)},  # preserve config epochs
    },
    "io": {"out_dir": "outputs_fever_nst_veri"},
}
if DEVICE in ("mps", "cpu"):
    VERI_OVERRIDES["model"] = {
        "name": "microsoft/deberta-v3-base",
        "use_lora": False,
        "gradient_checkpointing": True,
        "max_length": 384,
    }

print("=" * 65)
print("  NST-VERI: DeBERTa-v3-large + LoRA + CEGIS Verification")
print("  Gold Evidence, Setting A")
print("=" * 65)
print(f"  Device    : {DEVICE_NAME}")
print(f"  Train set : {n_train_str}")
print(f"  Epochs    : 5 (3-phase: NLI→contrastive→constraints)")
print(f"  Model     : DeBERTa-v3-large + LoRA (rank=16)")
print("=" * 65 + "\n")

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri = train_fever_veri("configs/fever_gold_nst_veri.yaml",
                                 config_overrides=VERI_OVERRIDES)
elapsed_veri = time.time() - t0

vdev = results_veri.get("dev", {})
vdt = results_veri.get("dev_test", {})
print(f"\n{'='*65}")
print(f"  NST-VERI RESULTS ({elapsed_veri/60:.1f} min)")
print(f"{'='*65}")
print(f"  Train size : {n_train_str}")
print(f"  Dev acc    : {vdev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {vdev.get('ece', 'N/A')}")
if vdt:
    print(f"  DevTest acc: {vdt.get('accuracy', 'N/A')}  (held-out)")
print(f"  Best dev   : {results_veri.get('best_dev_acc', 'N/A')}")
print(f"  Temperature: {results_veri.get('temperature', 'N/A')}")
viol = results_veri.get("constraint_violations", {})
if viol:
    print(f"  Violations : {viol}")
print(f"\n  Per-label (dev):")
for label, stats in vdev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_veri.json", "w") as f:
    json.dump(results_veri, f, indent=2, default=str)
print(f"\n  Saved to results_veri.json")

gc.collect()

In [ ]:
# ============================================================
# Cell 6: Honest Results Comparison
# ============================================================
import json, os
from datetime import datetime

# ── Gather all results ──
all_results = {}
for name, path in [("smoke", "results_smoke.json"),
                    ("neural_baseline", "results_baseline.json"),
                    ("nst_veri", "results_veri.json")]:
    if os.path.exists(path):
        with open(path) as f:
            all_results[name] = json.load(f)

print("=" * 72)
print("  HONEST RESULTS COMPARISON")
print(f"  Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"  Device: {DEVICE_NAME}")
print("=" * 72)

# ── Results table ──
header = f"{'Model':<25} {'Dev Acc':>10} {'Dev ECE':>10} {'DevTest':>10} {'Time':>10}"
print(f"\n{header}")
print("-" * 72)

for name, res in all_results.items():
    dev = res.get("dev", {})
    dt = res.get("dev_test", {})
    acc = dev.get("accuracy", "—")
    ece = dev.get("ece", "—")
    dt_acc = dt.get("accuracy", "—") if dt else "—"
    elapsed = res.get("elapsed_min", "—")
    if isinstance(acc, float): acc = f"{acc:.4f}"
    if isinstance(ece, float): ece = f"{ece:.4f}"
    if isinstance(dt_acc, float): dt_acc = f"{dt_acc:.4f}"
    if isinstance(elapsed, (int, float)): elapsed = f"{elapsed:.1f}m"
    print(f"{name:<25} {acc:>10} {ece:>10} {dt_acc:>10} {elapsed:>10}")

print("-" * 72)

# ── Honest assessment ──
if "neural_baseline" in all_results and "nst_veri" in all_results:
    b_acc = all_results["neural_baseline"].get("dev", {}).get("accuracy")
    v_acc = all_results["nst_veri"].get("dev", {}).get("accuracy")
    if isinstance(b_acc, (int, float)) and isinstance(v_acc, (int, float)):
        delta = v_acc - b_acc
        print(f"\n  NST-VERI vs Baseline delta: {delta:+.4f}")
        if delta > 0.01:
            print("  → NST-VERI shows improvement over neural baseline")
        elif delta < -0.01:
            print("  → NST-VERI underperforms neural baseline (constraints may be too aggressive)")
        else:
            print("  → Results are within noise margin (~1%). No clear winner.")
    b_ece = all_results["neural_baseline"].get("dev", {}).get("ece")
    v_ece = all_results["nst_veri"].get("dev", {}).get("ece")
    if isinstance(b_ece, (int, float)) and isinstance(v_ece, (int, float)):
        ece_delta = v_ece - b_ece
        print(f"  Calibration delta (ECE): {ece_delta:+.4f}")
        if ece_delta < -0.005:
            print("  → NST-VERI is better calibrated (lower ECE is better)")
        elif ece_delta > 0.005:
            print("  → Baseline is better calibrated")
        else:
            print("  → Calibration is similar")

# ── Caveats ──
print(f"\n  CAVEATS:")
if DEVICE == "mps":
    print(f"  • Trained on Apple Silicon MPS (not A100)")
    print(f"  • Limited to {MAX_TRAIN or '???'} training samples")
    print(f"  • FP16 mixed precision (not bf16)")
elif DEVICE == "cpu":
    print(f"  • Trained on CPU — results are preliminary at best")
print(f"  • Gold evidence setting (IR retrieval not tested)")
print(f"  • Single seed — no variance estimate")
print(f"  • Max length 384 (some evidence may be truncated)")

# ── Save consolidated report ──
report = {
    "date": datetime.now().isoformat(),
    "device": DEVICE_NAME,
    "results": all_results,
}
with open("results_consolidated.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
print(f"\n  Consolidated report saved to results_consolidated.json")

In [ ]:
# ============================================================
# Cell 9: 3K NEURAL BASELINE (Fair Comparison)
# ============================================================
# Same DeBERTa-v3-large + LoRA as NST-VERI, but NO constraints.
# This establishes the ceiling that pure neural achieves on 3k.
import time, json, gc

# Clear module cache for fresh imports
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 65)
print("  3K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_3k = train_fever_nst("configs/fever_neural_smoke_3k.yaml")
elapsed = time.time() - t0

dev = results_neural_3k.get("dev", {})
print(f"\n{'='*65}")
print(f"  NEURAL BASELINE 3K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural_3k.json", "w") as f:
    json.dump(results_neural_3k, f, indent=2, default=str)
print(f"  Saved to results_neural_3k.json")

In [ ]:
# ============================================================
# Cell 10: 3K NST-VERI SMOKE (Rapid Iteration — THE CRITICAL RUN)
# ============================================================
# Key changes from previous failed VERI run:
#   1. lambda_max: 0.3 → 2.0 (constraint loss was drowned out)
#   2. Adaptive lambda init: sigmoid(-3)≈0.05 → sigmoid(0)=0.5
#   3. Phase schedule: constraints now start epoch 2 (not epoch 2 of 5)
#   4. Constraint directions: sharper (0.85 peak vs 0.65)
#   5. Focal loss: enabled (focuses on hard examples)
#   6. Full constraint activity logging
#
# EXPECTED: constraint_loss > 0, fire_rate > 0, λ > 0.1
# If these are zero, the method is still broken.
import time, json, gc

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 65)
print("  3K NST-VERI SMOKE: Constraint-Enhanced Training")
print("  lambda_max=2.0, warm init, early constraints, focal loss")
print("=" * 65)

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri_3k = train_fever_veri("configs/fever_veri_smoke_3k.yaml")
elapsed = time.time() - t0

dev = results_veri_3k.get("dev", {})
print(f"\n{'='*65}")
print(f"  NST-VERI 3K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_veri_3k.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Constraint activity analysis
train_log = results_veri_3k.get("train_log", [])
if train_log:
    phase3_entries = [e for e in train_log if e.get("phase", 0) >= 3]
    if phase3_entries:
        cst_losses = [e.get("loss_constraint", 0) for e in phase3_entries]
        lambdas = [e.get("mean_lambda", 0) for e in phase3_entries]
        print(f"\n  CONSTRAINT DIAGNOSTICS:")
        print(f"    Phase 3 entries: {len(phase3_entries)}")
        print(f"    Constraint loss: min={min(cst_losses):.4f} max={max(cst_losses):.4f} mean={sum(cst_losses)/len(cst_losses):.4f}")
        print(f"    Mean lambda:     min={min(lambdas):.4f} max={max(lambdas):.4f} mean={sum(lambdas)/len(lambdas):.4f}")
        if max(cst_losses) > 0.001:
            print(f"    ✓ CONSTRAINTS ARE ACTIVE")
        else:
            print(f"    ✗ CONSTRAINTS STILL INACTIVE — needs further investigation")
    else:
        print(f"\n  WARNING: No Phase 3 entries in training log")

# Constraint calibration
calib = results_veri_3k.get("constraint_calibration", {})
if calib:
    print(f"\n  CONSTRAINT CALIBRATION (on dev):")
    for cname, cstats in calib.items():
        print(f"    {cname}: precision={cstats.get('precision', 0):.3f} fire_rate={cstats.get('fire_rate', 0):.3f}")

with open("results_veri_3k.json", "w") as f:
    json.dump(results_veri_3k, f, indent=2, default=str)
print(f"\n  Saved to results_veri_3k.json")

In [ ]:
# ============================================================
# Cell 11: 3K COMPARISON — Neural vs NST-VERI
# ============================================================
import json, os

experiments = {}
for name, path in [("Neural 3k", "results_neural_3k.json"),
                    ("NST-VERI 3k", "results_veri_3k.json")]:
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)

if len(experiments) >= 2:
    print("=" * 70)
    print("  3K SMOKE COMPARISON  — FEVER Gold Evidence")
    print("=" * 70)
    print(f"  {'Method':<20} {'Dev Acc':>8} {'ECE':>7} {'Brier':>7}")
    print("-" * 50)
    for name, r in experiments.items():
        dev = r.get("dev", {})
        acc = dev.get("accuracy", 0)
        ece = dev.get("ece", 0)
        brier = dev.get("brier", 0)
        print(f"  {name:<20} {acc:>8.4f} {ece:>7.4f} {brier:>7.4f}")
    print("-" * 50)

    n_acc = experiments["Neural 3k"]["dev"]["accuracy"]
    v_acc = experiments["NST-VERI 3k"]["dev"]["accuracy"]
    delta = v_acc - n_acc
    print(f"\n  Delta (VERI - Neural): {delta:+.4f}")
    if delta > 0.01:
        print(f"  → VERI shows real improvement (+{delta*100:.1f}%)")
        print(f"  → Proceed to full-scale run")
    elif delta > -0.01:
        print(f"  → Roughly tied — constraints may help on larger data")
    else:
        print(f"  → Neural wins — constraints hurting, need revision")

    # Per-label
    print(f"\n  PER-LABEL (Dev):")
    for label in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
        row = f"    {label:<22}"
        for name, r in experiments.items():
            per = r.get("dev", {}).get("per_label", {})
            if label in per:
                row += f"  {per[label].get('accuracy', 0):.4f}"
            else:
                row += f"  —"
        print(row)
else:
    print("Run cells 9 and 10 first to generate results")

In [ ]:
# ============================================================
# Cell 12: FULL NST-VERI RUN (Only if 3k shows signal)
# ============================================================
# Run this ONLY after Cell 11 shows VERI > Neural on 3k.
# Full ~145k train, ~19k dev, 8 epochs on A100.
# Expected runtime: ~40-60 min on A100.
import time, json, gc

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 65)
print("  FULL NST-VERI: DeBERTa-v3-large + LoRA + Constraints")
print("  8 epochs, lambda_max=2.0, focal loss, warm adaptive lambda")
print("=" * 65)

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri_full = train_fever_veri("configs/fever_gold_nst_veri.yaml")
elapsed = time.time() - t0

dev = results_veri_full.get("dev", {})
dt = results_veri_full.get("dev_test", {})
print(f"\n{'='*65}")
print(f"  FULL NST-VERI RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
if dt:
    print(f"  DevTest acc: {dt.get('accuracy', 'N/A')}  (held-out)")
print(f"  Best dev   : {results_veri_full.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Constraint diagnostics
train_log = results_veri_full.get("train_log", [])
phase3_entries = [e for e in train_log if e.get("phase", 0) >= 3]
if phase3_entries:
    cst = [e.get("loss_constraint", 0) for e in phase3_entries]
    lam = [e.get("mean_lambda", 0) for e in phase3_entries]
    print(f"\n  CONSTRAINT DIAGNOSTICS:")
    print(f"    Constraint loss: mean={sum(cst)/len(cst):.4f}")
    print(f"    Mean lambda: mean={sum(lam)/len(lam):.4f}")

calib = results_veri_full.get("constraint_calibration", {})
if calib:
    print(f"\n  CONSTRAINT CALIBRATION:")
    for cname, cstats in calib.items():
        print(f"    {cname}: precision={cstats.get('precision', 0):.3f} fire_rate={cstats.get('fire_rate', 0):.3f}")

with open("results_veri_full.json", "w") as f:
    json.dump(results_veri_full, f, indent=2, default=str)
print(f"\n  Saved to results_veri_full.json")

In [ ]:
# ============================================================
# Cell 13: FULL NEURAL BASELINE (Fair Comparison)
# ============================================================
# Same DeBERTa-v3-large + LoRA, same epochs as full VERI run.
# Run after Cell 12 for fair comparison.
import time, json, gc

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 65)
print("  FULL NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_full = train_fever_nst("configs/fever_gold_neural_large.yaml")
elapsed = time.time() - t0

dev = results_neural_full.get("dev", {})
dt = results_neural_full.get("dev_test", {})
print(f"\n{'='*65}")
print(f"  NEURAL BASELINE FULL RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
if dt:
    print(f"  DevTest acc: {dt.get('accuracy', 'N/A')}  (held-out)")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural_full.json", "w") as f:
    json.dump(results_neural_full, f, indent=2, default=str)
print(f"  Saved to results_neural_full.json")

In [ ]:
# ============================================================
# Cell 14: FINAL COMPARISON & SAVE
# ============================================================
import json, os, glob

experiments = {}
for name, path in [
    ("Neural 3k",       "results_neural_3k.json"),
    ("NST-VERI 3k",     "results_veri_3k.json"),
    ("Neural Full",     "results_neural_full.json"),
    ("NST-VERI Full",   "results_veri_full.json"),
]:
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)

print("=" * 80)
print("  ALL RESULTS — FEVER Gold-Evidence Label Accuracy")
print("=" * 80)
print(f"  {'Method':<20} {'Dev Acc':>8} {'DevTest':>8} {'ECE':>7} {'Brier':>7} {'Scale':<10}")
print("-" * 70)
for name, r in experiments.items():
    dev = r.get("dev", {})
    dt = r.get("dev_test", {})
    acc = dev.get("accuracy", 0)
    dt_acc = dt.get("accuracy", 0) if dt else 0
    ece = dev.get("ece", 0)
    brier = dev.get("brier", 0)
    scale = "3k" if "3k" in name else "full"
    print(f"  {name:<20} {acc:>8.4f} {dt_acc:>8.4f} {ece:>7.4f} {brier:>7.4f} {scale:<10}")
print("-" * 70)

# Final verdict
if "NST-VERI Full" in experiments and "Neural Full" in experiments:
    v = experiments["NST-VERI Full"]["dev"]["accuracy"]
    n = experiments["Neural Full"]["dev"]["accuracy"]
    d = v - n
    print(f"\n  FAIR COMPARISON (same DeBERTa-v3-large + LoRA):")
    print(f"    NST-VERI: {v:.4f}  Neural: {n:.4f}  Delta: {d:+.4f}")
    if d > 0.005:
        print(f"    VERDICT: NST-VERI WINS (+{d*100:.2f}%)")
    elif d < -0.005:
        print(f"    VERDICT: Neural wins (constraints hurt)")
    else:
        print(f"    VERDICT: Tied (constraint effect is negligible)")

    if v >= 0.90:
        print(f"\n  ★ 90%+ FEVER gold-evidence label accuracy ACHIEVED: {v:.4f}")
    else:
        print(f"\n  ✗ 90% target not reached. Best: {v:.4f}")

# List artifacts
print(f"\n  ARTIFACTS:")
for f in sorted(glob.glob("results_*.json")):
    print(f"    {f} ({os.path.getsize(f):,} bytes)")
for d in sorted(glob.glob("outputs_*")):
    if os.path.isdir(d):
        print(f"    {d}/ ({len(os.listdir(d))} files)")